# Sensor Air Quality Index (AQI) calculation for the past 1h, 8h or 24h

#### Before starting run the next cell to install the required Python libraries first required to run the code:

In [ ]:
# run this cell once only if you need to install the libraries
!pip install requests pandas plotly matplotlib seaborn
!pip install --upgrade pip

### **Step 1: Get Sensor Data: Request your sensor device last observations from the database (API).**
#### You need to run the next cell and you will be asked to:
- Enter Smart Citizen device ID:
- Select the time to use to calculate the AQI: "1 - last 1 hour"; "2 - last 8 hours";  "3 - last 24 hours"
- Select the Observed Property: "1 - Particle Matter PM 2.5"; "2 - Carbon Dioxide"

In [ ]:
# Run this cell to be asked for your input below
import requests
import datetime
import pandas as pd
import plotly.express as px
import seaborn as sns
from IPython.display import display, HTML

def get_sensor_data():
    try:
        # 1. Device ID input
        device_id = input("Enter your Smart Citizen Device ID: ").strip()
        # 17277 - Blackrock, Dublin, Ireland
        # 17183 - Dalkey, Ireland
        if not device_id:
            raise ValueError("Device ID cannot be empty.")
        
        # 2. Time range selection
        print("\nSelect the time range to retrieve data:")
        print("1 - Last 1 hour")
        print("2 - Last 8 hours")
        print("3 - Last 24 hours")
        time_choice = input("Enter 1, 2, or 3: ").strip()
        
        time_mapping = {"1": 1, "2": 8, "3": 24}
        hours_back = time_mapping.get(time_choice)
        if not hours_back:
            raise ValueError("Invalid choice for time range. Please enter 1, 2, or 3.")
        
        # 3. Observed property selection
        print("\nSelect the observed property:")
        print("1 - PM 2.5 (Particulate Matter)")
        print("2 - CO₂ (Carbon Dioxide)")
        sensor_choice = input("Enter 1 or 2: ").strip()
        
        sensor_mapping = {"1": 87, "2": 158}
        observed_property_mapping = {"1": "PM2.5", "2": "CO₂"}
        sensor_id = sensor_mapping.get(sensor_choice)
        observed_property_name = observed_property_mapping.get(sensor_choice)
        if not sensor_id or not observed_property_name:
            raise ValueError("Invalid choice for observed property. Please enter 1 or 2.")
        
        # 4. Define time window
        now = datetime.datetime.utcnow()
        from_time = now - datetime.timedelta(hours=hours_back)
        from_time_str = from_time.strftime('%Y-%m-%dT%H:%M:%S')
        to_time_str = now.strftime('%Y-%m-%dT%H:%M:%S')
        time_interval_str = f"{hours_back}h"

        # 5. Fetch data from API
        api_url = (
            f"https://api.smartcitizen.me/v0/devices/{device_id}/readings?"
            f"sensor_id={sensor_id}&rollup=5m&from={from_time_str}&to={to_time_str}"
        )
        print(f"\nFetching data from: {api_url}")
        response = requests.get(api_url)
        response.raise_for_status()
        data = response.json()
        
        # 6. Parse readings
        observations = data.get("readings", [])
        if not observations:
            raise ValueError("No observations found for the selected parameters.")
        
        df = pd.DataFrame(observations, columns=["datetime", "value"])
        df["datetime"] = pd.to_datetime(df["datetime"])
        observations_avg = df["value"].mean()

        # 7. Display table
        sns.set_style("whitegrid")
        display(HTML(f"""
        <style>
            .table-container {{
                width: 100%;
                display: flex;
                justify-content: center;
            }}
            .table-striped {{
                width: 100%;
                max-width: 100%;
            }}
        </style>
        <div class='table-container'>
            {df.to_html(classes='table table-striped', escape=False, border=0)}
        </div>
        """))

        # 8. Plot chart
        fig = px.line(df, x='datetime', y='value',
                      title=f"{observed_property_name} Readings Over the Last {hours_back} Hours",
                      markers=True)
        fig.update_xaxes(title_text='Datetime', tickangle=-45)
        fig.update_yaxes(title_text='Value')
        fig.show()

        # 9. Display average
        display(HTML(
            f"<h3>Average {observed_property_name} value over the last {time_interval_str}: "
            f"<strong>{observations_avg:.2f}</strong></h3>"
        ))

        return df, observed_property_name, observations_avg, time_interval_str, from_time_str, to_time_str

    except requests.exceptions.RequestException as e:
        print(f"\nNetwork error: {e}")
    except ValueError as e:
        print(f"\nInput error: {e}")
    except Exception as e:
        print(f"\nUnexpected error: {e}")

    return None, None, None, None, None, None

# Run the function
readings_df, observed_property_name, observations_avg, time_interval_str, from_time_str, to_time_str = get_sensor_data()

### **Step 2: Air Quality Index calculation using the sensor observations retrieved before.**
#### You need to run the next cell and you will be asked to:
- Select the guideline you want to use for the AQI standard: "1 - US EPA"; "2 - WHO"

In [ ]:
import numpy as np
import pandas as pd

# USEPA AQI Data
usepa_data = {
    'Rating': ['Excellent', 'Fine', 'Moderate', 'Poor', 'Very Poor', 'Severe'],
    'Index': [1, 2, 3, 4, 5, 6],
    'AQI_value_low': [0, 51, 101, 151, 201, 301],
    'AQI_value_high': [50, 100, 150, 200, 300, 400],
    'CO2_low': [250, 401, 1001, 1501, 2001, 5000],
    'CO2_high': [400, 1000, 1500, 2000, 5000, float('inf')],
    'PM2.5_low': [0, 11, 21, 26, 51, 76],
    'PM2.5_high': [10, 20, 25, 50, 75, 800],
    'Rating_colors': ['#AED579', '#4F8E62', '#F6C546', '#df851f', '#ea3525', '#702218'],
}

# TwinAIR AQI Data
who_data = {
    'Rating': ['Good', 'Moderate', 'High Risk'],
    'Index_low': [0, 3, 7],
    'Index_high': [2, 6, 10],
    'PM2.5_low': [0, 15, 36],
    'PM2.5_high': [15, 35, 400],
    'CO2_low': [400, 801, 1401],
    'CO2_high': [800, 1400, 5000],
    'Rating_colors': ['#a8e6a3', '#fff59d', '#ff8a80'],
}

# Convert to DataFrames
df_usepa = pd.DataFrame(usepa_data)
df_who = pd.DataFrame(who_data)

def calculate_aq_index(pollutant_name, observation_input, guideline='USEPA', time_interval='1h'):
    df = df_usepa if guideline.upper() == 'USEPA' else df_who
    low_col = f'{pollutant_name}_low'
    high_col = f'{pollutant_name}_high'
    
    try:
        range_row = df[(df[low_col] <= observation_input) & (df[high_col] >= observation_input)].iloc[0]
    except IndexError:
        return {'pollutant': pollutant_name, 'aqi_index': None, 'index': None, 'rating': "Out of range", 'rating_color': '#FFFFFF'}
    
    # aqi_index = round(range_row['AQI_value_low'] + (observation_input - range_row[low_col]) *
      #                ((range_row['AQI_value_high'] - range_row['AQI_value_low']) /
      #                (range_row[high_col] - range_row[low_col])), 1)

    # AQI Calculation using linear interpolation
    obs_low = range_row[low_col]
    obs_high = range_row[high_col]
    if guideline.upper() == 'USEPA':
        aqi_low = range_row['AQI_value_low']
        aqi_high = range_row['AQI_value_high']
        index = range_row['Index']  # Corrected index extraction
    else:  # WHO uses Index_low and Index_high
        aqi_low = range_row['Index_low']
        aqi_high = range_row['Index_high']
        index = int(aqi_high)  # WHO logic remains unchanged

    # Calculate AQI using the provided formula
    if np.isinf(obs_high) or pd.isna(obs_high):
        aqi_index = aqi_high
    else:
        aqi_index = (((aqi_high - aqi_low) / (obs_high - obs_low)) *
                     (observation_input - obs_low) + aqi_low)

    aqi_index = round(aqi_index, 1)  # Round AQI to one decimal
    
    return {
        'pollutant': pollutant_name,
        'aqi_index': aqi_index,
        'index': range_row['Index'] if 'Index' in range_row else None,  # ✅ Include the AQI index
        'rating': range_row['Rating'],
        'rating_color': range_row['Rating_colors'] if 'Rating_colors' in range_row else '#FFFFFF'
    }


In [ ]:
# Ask the user for AQI guideline selection
print("Select the AQI Guideline:")
print("1 - USEPA")
print("2 - WHO")
guideline_choice = input("Enter 1 or 2: ")

guideline_mapping = {"1": "USEPA", "2": "WHO"}
guideline = guideline_mapping.get(guideline_choice)

if not guideline:
    raise ValueError("Invalid choice. Please enter 1 or 2.")

# Ensure all required values are available
if observed_property_name and observations_avg and guideline and time_interval_str:
    aqi_result = calculate_aq_index(observed_property_name, observations_avg, guideline, time_interval_str)

    # Create a DataFrame for the AQI results
    aqi_df = pd.DataFrame([{
        "Pollutant": observed_property_name,
        "Input Value": round(observations_avg, 2),
        "Index": aqi_result["index"],
        "Rating": aqi_result["rating"],
        "Time Interval": time_interval_str,
        "Start (datetime)": from_time_str,
        "End (datetime)": to_time_str
    }])

    # Apply background color styling based on AQI rating
    styled_aqi_table = aqi_df.style.map(
        lambda x: f'background-color: {aqi_result["rating_color"]}; color: black;', 
        subset=pd.IndexSlice[:, :]
    )

    # Display the styled DataFrame
    display(HTML(styled_aqi_table.to_html()))
else:
    print("Missing required values from the first cell. Please run it first.")